# Explainability in CLARYON

Demonstrates SHAP and LIME explanations on classical and quantum models.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X = iris.data
y = (iris.target > 0).astype(int)
feature_names = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
# Train a classical MLP
from claryon.models.classical.mlp_ import MLPModel
from claryon.io.base import TaskType

mlp = MLPModel(seed=42)
mlp.fit(X_train, y_train, TaskType.BINARY)
preds = mlp.predict(X_test)
print(f"MLP accuracy: {np.mean(preds == y_test):.3f}")

In [ ]:
# SHAP explanations
try:
    from claryon.explainability.shap_ import SHAPExplainer

    shap_exp = SHAPExplainer(max_features=4, max_test_samples=5)
    result = shap_exp.explain(
        mlp.predict_proba, X_test, feature_names=feature_names, X_train=X_train
    )
    print("SHAP values shape:", result["shap_values"].shape)
    
    # Show feature importance
    mean_abs_shap = np.abs(result["shap_values"]).mean(axis=0)
    for fname, val in sorted(zip(feature_names, mean_abs_shap), key=lambda x: -x[1]):
        print(f"  {fname}: {val:.4f}")
except ImportError:
    print("shap not installed — pip install shap")

In [ ]:
# LIME explanations
try:
    from claryon.explainability.lime_ import LIMEExplainer

    lime_exp = LIMEExplainer(max_features=4, max_test_samples=3)
    result = lime_exp.explain(
        mlp.predict_proba, X_test, feature_names=feature_names, X_train=X_train
    )
    print(f"LIME explanations for {len(result['explanations'])} samples")
    for i, exp in enumerate(result["explanations"][:2]):
        print(f"\nSample {i}: {exp}")
except ImportError:
    print("lime not installed — pip install lime")